# CSE382 — Mushroom Edibility Classification
**First Deliverable**

Full documentation is provided in the attached report.

## 0. Setup and reproducibility contract

In [1]:
import sys, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, make_scorer, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import (StratifiedKFold, cross_val_predict, cross_val_score,
                                     cross_validate, learning_curve, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TEST_SIZE    = 0.20
N_FOLDS      = 5
CV = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
np.random.seed(RANDOM_STATE)

FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
                     "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold"})

EDIBLE, POISON, GREY = "#2a9d8f", "#e76f51", "#8d99ae"

print("Environment")
print(f"  python       {sys.version.split()[0]}  ({platform.system()})")
print(f"  numpy        {np.__version__}")
print(f"  pandas       {pd.__version__}")
print(f"  scikit-learn {sklearn.__version__}")
print(f"  matplotlib   {plt.matplotlib.__version__}")
print(f"  seaborn      {sns.__version__}")
print(f"\nSeed = {RANDOM_STATE} | test size = {TEST_SIZE:.0%} | CV = {N_FOLDS}-fold stratified")

Environment
  python       3.12.13  (Linux)
  numpy        2.0.2
  pandas       2.2.2
  scikit-learn 1.6.1
  matplotlib   3.10.0
  seaborn      0.13.2

Seed = 42 | test size = 20% | CV = 5-fold stratified


## 1. Data acquisition and documentation

### 1.1 Provisioning (idempotent: reuse → download → upload fallback)

In [2]:
import io, zipfile, urllib.request

DATA_FILE = Path("agaricus-lepiota.data")
UCI_ZIP   = "https://archive.ics.uci.edu/static/public/73/mushroom.zip"

def have_data() -> bool:
    return DATA_FILE.exists() and DATA_FILE.stat().st_size > 300_000

if have_data():
    print(f"Found {DATA_FILE} locally ({DATA_FILE.stat().st_size:,} bytes) — reusing it.")
else:
    print("Dataset not found locally. Downloading from the UCI repository ...")
    try:
        req = urllib.request.Request(UCI_ZIP, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=90) as resp:
            payload = resp.read()
        zipfile.ZipFile(io.BytesIO(payload)).extractall(".")
        print(f"Downloaded and extracted ({len(payload):,} bytes).")
    except Exception as exc:
        print(f"Download failed: {type(exc).__name__}: {exc}")
        try:
            from google.colab import files
            print("\nUpload mushroom.zip (or agaricus-lepiota.data) using the picker below.")
            for fname in files.upload():
                if fname.lower().endswith(".zip"):
                    zipfile.ZipFile(fname).extractall(".")
                    print(f"Extracted {fname}")
        except ImportError:
            raise FileNotFoundError(
                "agaricus-lepiota.data not found and could not be downloaded. "
                "Place it next to this notebook and re-run.") from exc

if not have_data():
    raise FileNotFoundError("Dataset still unavailable after provisioning.")
print(f"\nReady: {DATA_FILE} ({DATA_FILE.stat().st_size:,} bytes)")

Dataset not found locally. Downloading from the UCI repository ...
Downloaded and extracted (141,318 bytes).

Ready: agaricus-lepiota.data (373,704 bytes)


### 1.2 Load and apply the schema

In [3]:
COLUMNS = [
    "class", "cap-shape", "cap-surface", "cap-color", "bruises", "odor",
    "gill-attachment", "gill-spacing", "gill-size", "gill-color",
    "stalk-shape", "stalk-root", "stalk-surface-above-ring",
    "stalk-surface-below-ring", "stalk-color-above-ring",
    "stalk-color-below-ring", "veil-type", "veil-color", "ring-number",
    "ring-type", "spore-print-color", "population", "habitat",
]

df = pd.read_csv(DATA_FILE, header=None, names=COLUMNS, dtype=str)

print(f"Shape             : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Duplicate rows    : {df.duplicated().sum()}")
print(f"Pandas-detected   : {df.isna().sum().sum()} nulls   <-- misleading, see section 2.2")
print(f"Literal '?' cells : {(df == '?').sum().sum():,}")
df.head()

Shape             : 8,124 rows x 23 columns
Duplicate rows    : 0
Pandas-detected   : 0 nulls   <-- misleading, see section 2.2
Literal '?' cells : 2,480


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


### 1.3 Integrity check against the published UCI specification

In [4]:
checks = {
    "8,124 instances (names §5)":           df.shape[0] == 8124,
    "22 attributes + 1 target (names §6)":  df.shape[1] == 23,
    "4,208 edible = 51.8% (names §9)":      (df["class"] == "e").sum() == 4208,
    "3,916 poisonous = 48.2% (names §9)":   (df["class"] == "p").sum() == 3916,
    "2,480 '?' in attribute 11 (names §8)": (df["stalk-root"] == "?").sum() == 2480,
}
for label, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {label}")
assert all(checks.values()), "Downloaded file does not match the UCI specification."
print("\nFile matches the published UCI specification on all five counts.")

  [PASS] 8,124 instances (names §5)
  [PASS] 22 attributes + 1 target (names §6)
  [PASS] 4,208 edible = 51.8% (names §9)
  [PASS] 3,916 poisonous = 48.2% (names §9)
  [PASS] 2,480 '?' in attribute 11 (names §8)

File matches the published UCI specification on all five counts.


### 1.4 Attribute profile

In [5]:
profile = pd.DataFrame({
    "n_levels":   df.nunique(),
    "top_level":  df.mode().iloc[0],
    "top_freq_%": (df.apply(lambda s: s.value_counts(normalize=True).iloc[0]) * 100).round(1),
    "n_unknown":  (df == "?").sum(),
})
profile

,n_levels,top_level,top_freq_%,n_unknown
class,2,e,51.8,0
cap-shape,6,x,45.0,0
cap-surface,4,y,39.9,0
cap-color,10,n,28.1,0
bruises,2,f,58.4,0
odor,9,n,43.4,0
gill-attachment,2,f,97.4,0
gill-spacing,2,c,83.9,0
gill-size,2,b,69.1,0
gill-color,12,b,21.3,0


## 2. Data cleaning and preprocessing

### 2.1 Train/test split — performed BEFORE any data-dependent decision

```text
The dataset is split before performing missingness analysis or feature-reduction decisions.
All data-dependent preprocessing decisions use the training set only.
```

In [6]:
y_all = (df["class"] == "p").astype(int)          # poisonous = 1 = positive class
X_all = df.drop(columns=["class"]).copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=RANDOM_STATE)

MAJORITY = y_train.value_counts(normalize=True).max()

print(f"Train : {X_train_raw.shape[0]:,} rows x {X_train_raw.shape[1]} cols "
      f"({y_train.mean():.2%} poisonous)")
print(f"Test  : {X_test_raw.shape[0]:,} rows x {X_test_raw.shape[1]} cols "
      f"({y_test.mean():.2%} poisonous)   [SEALED until section 5.3]")
print(f"\nStratification drift : {abs(y_train.mean() - y_test.mean()):.4%}")
print(f"Majority-class floor : {MAJORITY:.4f}")

Train : 6,499 rows x 22 cols (48.21% poisonous)
Test  : 1,625 rows x 22 cols (48.18% poisonous)   [SEALED until section 5.3]

Stratification drift : 0.0228%
Majority-class floor : 0.5179


### 2.2 Missing values: is `?` associated with the target?

In [7]:
unknown_train = X_train_raw["stalk-root"] == "?"
print(f"Unknown stalk-root in TRAIN: {unknown_train.sum():,} rows ({unknown_train.mean():.1%})\n")

tab = pd.crosstab(unknown_train, y_train, normalize="index").round(4) * 100
tab.index   = ["stalk-root recorded", "stalk-root unknown"]
tab.columns = ["% edible", "% poisonous"]
tab["n"] = pd.crosstab(unknown_train, y_train).sum(axis=1).values
print(tab.round(1).to_string())

base = y_train.mean() * 100
print(f"\nOverall poisonous rate in TRAIN : {base:.1f}%")
print(f"Lift when stalk-root is unknown : "
      f"{tab.loc['stalk-root unknown', '% poisonous'] - base:+.1f} pp")

Unknown stalk-root in TRAIN: 1,968 rows (30.3%)

                     % edible  % poisonous     n
stalk-root recorded      61.6         38.4  4531
stalk-root unknown       29.2         70.8  1968

Overall poisonous rate in TRAIN : 48.2%
Lift when stalk-root is unknown : +22.6 pp


### 2.3 Apply the retain-as-a-level policy

In [8]:
def normalise_unknowns(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["stalk-root"] = out["stalk-root"].replace("?", "unknown")
    return out

X_train_raw = normalise_unknowns(X_train_raw)
X_test_raw  = normalise_unknowns(X_test_raw)

print("stalk-root levels after normalisation:", sorted(X_train_raw["stalk-root"].unique()))
print("Remaining '?' cells in train / test  :",
      (X_train_raw == "?").sum().sum(), "/", (X_test_raw == "?").sum().sum())

stalk-root levels after normalisation: ['b', 'c', 'e', 'r', 'unknown']
Remaining '?' cells in train / test  : 0 / 0


### 2.4 Feature-reduction evidence — two measures, training set only

In [9]:
ord_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train_ord = pd.DataFrame(ord_encoder.fit_transform(X_train_raw), columns=X_train_raw.columns)

mi_scores = mutual_info_classif(X_train_ord, y_train,
                                discrete_features=True, random_state=RANDOM_STATE)

records = []
for j, col in enumerate(X_train_raw.columns):
    solo_acc = cross_val_score(DecisionTreeClassifier(random_state=RANDOM_STATE),
                               X_train_ord[[col]], y_train, cv=CV, scoring="accuracy").mean()
    records.append({"feature": col, "n_levels": X_train_raw[col].nunique(),
                    "solo_accuracy": solo_acc, "mutual_info": mi_scores[j]})

solo_df = (pd.DataFrame(records).sort_values("solo_accuracy", ascending=False)
             .reset_index(drop=True))
solo_df["solo_rank"]   = solo_df.index + 1
solo_df["mi_rank"]     = solo_df["mutual_info"].rank(ascending=False).astype(int)
solo_df["gap_to_next"] = solo_df["solo_accuracy"].diff(-1)
solo_df.round(4)

,feature,n_levels,solo_accuracy,mutual_info,solo_rank,mi_rank,gap_to_next
0,odor,9,0.9851,0.6275,1,1,0.1159
1,spore-print-color,9,0.8692,0.3368,2,2,0.0649
2,gill-color,12,0.8043,0.2885,3,3,0.0275
3,stalk-surface-above-ring,4,0.7767,0.2001,4,5,0.0015
4,ring-type,5,0.7752,0.2213,5,4,0.0078
5,stalk-surface-below-ring,4,0.7674,0.1893,6,6,0.0154
6,gill-size,2,0.7520,0.1550,7,9,0.0074
7,bruises,2,0.7446,0.1339,8,11,0.0226
8,population,6,0.7220,0.1413,9,10,0.0043
9,stalk-color-above-ring,9,0.7176,0.1774,10,7,0.0018


### 2.5 Selection: top-3 under both measures, plus zero-variance removal

In [10]:
TOP_K = 3   # the three strongest individual predictors

top_solo = solo_df.nlargest(TOP_K, "solo_accuracy")["feature"].tolist()
top_mi   = solo_df.nlargest(TOP_K, "mutual_info")["feature"].tolist()

print(f"Top-{TOP_K} by solo accuracy    : {top_solo}")
print(f"Top-{TOP_K} by mutual information: {top_mi}")
print(f"The two measures agree          : {set(top_solo) == set(top_mi)}")

SHORTCUT_FEATURES = top_solo
CONSTANT_FEATURES = [c for c in X_train_raw.columns if X_train_raw[c].nunique() == 1]
DROPPED = SHORTCUT_FEATURES + CONSTANT_FEATURES

print(f"\nAccuracy gap between rank {TOP_K} and rank {TOP_K + 1}: "
      f"{solo_df.loc[TOP_K - 1, 'solo_accuracy'] - solo_df.loc[TOP_K, 'solo_accuracy']:.4f}")
print(f"Accuracy gap between rank {TOP_K + 1} and rank {TOP_K + 2}: "
      f"{solo_df.loc[TOP_K, 'solo_accuracy'] - solo_df.loc[TOP_K + 1, 'solo_accuracy']:.4f}")
print(f"\nShortcut features : {SHORTCUT_FEATURES}")
print(f"Zero-variance     : {CONSTANT_FEATURES}")

Top-3 by solo accuracy    : ['odor', 'spore-print-color', 'gill-color']
Top-3 by mutual information: ['odor', 'spore-print-color', 'gill-color']
The two measures agree          : True

Accuracy gap between rank 3 and rank 4: 0.0275
Accuracy gap between rank 4 and rank 5: 0.0015

Shortcut features : ['odor', 'spore-print-color', 'gill-color']
Zero-variance     : ['veil-type']


### 2.6 Sensitivity of the cut (supporting evidence)

In [11]:
def make_pipeline(model):
    """One-hot encoding is fitted INSIDE every CV fold — never on the full training set."""
    return Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                     ("clf", model)])

sens = []
for k in [0, 2, 3, 4, 5]:
    drop_k = solo_df["feature"][:k].tolist() + CONSTANT_FEATURES
    Xk = X_train_raw.drop(columns=drop_k)
    stump = cross_val_score(make_pipeline(DecisionTreeClassifier(max_depth=1,
                            random_state=RANDOM_STATE)), Xk, y_train, cv=CV).mean()
    linear = cross_val_score(make_pipeline(LogisticRegression(max_iter=5000,
                             random_state=RANDOM_STATE)), Xk, y_train, cv=CV).mean()
    best_left = solo_df.loc[~solo_df["feature"].isin(drop_k), "solo_accuracy"].max()
    sens.append({"shortcuts_dropped": k, "features_kept": Xk.shape[1],
                 "best_remaining_solo": round(best_left, 4),
                 "depth1_tree_cv_acc": round(stump, 4),
                 "logistic_cv_acc": round(linear, 4),
                 "note": "<-- adopted" if k == TOP_K else ""})
pd.DataFrame(sens)

,shortcuts_dropped,features_kept,best_remaining_solo,depth1_tree_cv_acc,logistic_cv_acc,note
0,0,21,0.9851,0.8871,0.9994,
1,2,19,0.8043,0.7767,0.9962,
2,3,18,0.7767,0.7767,0.9935,<-- adopted
3,4,17,0.7752,0.7674,0.9922,
4,5,16,0.7674,0.7674,0.9898,


### 2.7 Apply the reduction

In [12]:
X_train = X_train_raw.drop(columns=DROPPED)
X_test  = X_test_raw.drop(columns=DROPPED)

print(f"Dropped {len(DROPPED)} attributes: {', '.join(DROPPED)}")
print(f"Retained {X_train.shape[1]} attributes\n")
print(f"Train feature matrix: {X_train.shape}")
print(f"Test  feature matrix: {X_test.shape}")
assert list(X_train.columns) == list(X_test.columns), "Train/test schema mismatch"
print("\nSchema consistency: OK")

Dropped 4 attributes: odor, spore-print-color, gill-color, veil-type
Retained 18 attributes

Train feature matrix: (6499, 18)
Test  feature matrix: (1625, 18)

Schema consistency: OK


### 2.8 Class balance

In [13]:
balance = y_train.value_counts().rename({0: "edible", 1: "poisonous"}).sort_index()
print(balance.to_string())
print(f"\nMinority share       : {balance.min() / balance.sum():.1%}")
print(f"Imbalance ratio      : 1 : {balance.max() / balance.min():.3f}")
print(f"Majority-class floor : {MAJORITY:.2%}")

class
edible       3366
poisonous    3133

Minority share       : 48.2%
Imbalance ratio      : 1 : 1.074
Majority-class floor : 51.79%


### 2.9 Encoding footprint

In [14]:
n_full    = OneHotEncoder(handle_unknown="ignore").fit_transform(X_train_raw).shape[1]
n_encoded = OneHotEncoder(handle_unknown="ignore").fit_transform(X_train).shape[1]
print(f"Before reduction: {X_train_raw.shape[1]} attributes -> {n_full} indicators")
print(f"After  reduction: {X_train.shape[1]} attributes -> {n_encoded} indicators")
print(f"Training rows per encoded column: {X_train.shape[0] / n_encoded:.0f}")

Before reduction: 22 attributes -> 117 indicators
After  reduction: 18 attributes -> 86 indicators
Training rows per encoded column: 76
